# Ames Housing Regression EDA

## Problem Statement
This notebook performs complete exploratory data analysis on a housing price regression dataset before any model building.

## What this notebook covers
- target distribution
- missing values
- duplicates
- numerical profiling with NumPy
- categorical profiling
- outliers
- skewness
- feature-target relationships
- preprocessing decisions
- cleaned CSV save

## Telugu
Regression model build cheyyadam mundu `SalePrice` target ni and features ni deep ga analyze cheyyali.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
from math import sqrt

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from scipy.stats import pearsonr, spearmanr

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

sns.set_theme(style="whitegrid")

os.makedirs("../data/raw", exist_ok=True)
os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../reports/figures", exist_ok=True)
os.makedirs("../reports", exist_ok=True)

## Folder structure expected

```text
eda-ames-housing/
├── data/
│   ├── raw/
│   │   └── train.csv
│   └── processed/
├── notebooks/
│   └── ames_regression_eda.ipynb
├── reports/
│   ├── figures/
│   └── insights.md
└── README.md
```

In [ ]:
df = pd.read_csv("../data/raw/train.csv")

print("Shape:", df.shape)
display(df.head())

In [ ]:
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
)

print(df.columns.tolist()[:20])

In [ ]:
print(df.info())
print("\n")
display(df.describe(include="all").T)

In [ ]:
target_col = "saleprice"
print("Target present:", target_col in df.columns)

## Why check target first?
Regression problem lo target mandatory.  
`SalePrice` unda leda first verify chestham.

In [ ]:
missing_report = (
    df.isna()
      .sum()
      .to_frame("missing_count")
      .assign(missing_pct=lambda x: (x["missing_count"] / len(df) * 100).round(2))
      .sort_values("missing_pct", ascending=False)
)

display(missing_report.head(40))
missing_report.to_csv("../data/processed/ames_missing_report.csv")

In [ ]:
plt.figure(figsize=(16, 8))
sns.heatmap(df.isnull(), cbar=False)
plt.title("Missing Values Heatmap")
plt.tight_layout()
plt.savefig("../reports/figures/missing_values_heatmap.png", dpi=200, bbox_inches="tight")
plt.show()

## Why missing heatmap?
Pattern-based missingness visual ga kanipisthundi.

In [ ]:
duplicate_count = df.duplicated().sum()
print("Duplicate rows:", duplicate_count)

In [ ]:
df = df.drop_duplicates().copy()
print("Shape after duplicate removal:", df.shape)

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()

print("Numerical columns:", len(num_cols))
print("Categorical columns:", len(cat_cols))
print("\nSample numerical:", num_cols[:15])
print("\nSample categorical:", cat_cols[:15])

# Target Analysis First

Regression lo target behavior chala important.
`SalePrice` skew unda? outliers unnaya? log transform useful aa?

In [ ]:
target = df[target_col].dropna().to_numpy()

target_summary = pd.DataFrame([{
    "target": target_col,
    "count": target.size,
    "mean": float(np.mean(target)),
    "median": float(np.median(target)),
    "std": float(np.std(target)),
    "var": float(np.var(target)),
    "min": float(np.min(target)),
    "max": float(np.max(target)),
    "p01": float(np.percentile(target, 1)),
    "p05": float(np.percentile(target, 5)),
    "p25": float(np.percentile(target, 25)),
    "p50": float(np.percentile(target, 50)),
    "p75": float(np.percentile(target, 75)),
    "p95": float(np.percentile(target, 95)),
    "p99": float(np.percentile(target, 99)),
    "skewness": float(pd.Series(target).skew())
}])

display(target_summary)
target_summary.to_csv("../data/processed/target_summary.csv", index=False)

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df[target_col], bins=40, kde=True)
plt.title("SalePrice Distribution")
plt.tight_layout()
plt.savefig("../reports/figures/saleprice_distribution.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(10, 3.5))
sns.boxplot(x=df[target_col])
plt.title("SalePrice Boxplot")
plt.tight_layout()
plt.show()

In [ ]:
df["log_saleprice"] = np.log1p(df[target_col])

plt.figure(figsize=(10, 5))
sns.histplot(df["log_saleprice"], bins=40, kde=True)
plt.title("Log1p(SalePrice) Distribution")
plt.tight_layout()
plt.show()

## Why log transform check?
House price target usually right-skewed untundi.  
`log1p` chesaka stable ga maruthunda ani chustham.

In [ ]:
numeric_summary = []

for col in num_cols:
    arr = df[col].dropna().to_numpy()
    if arr.size == 0:
        continue

    numeric_summary.append({
        "column": col,
        "count": arr.size,
        "mean": float(np.mean(arr)),
        "median": float(np.median(arr)),
        "std": float(np.std(arr)),
        "var": float(np.var(arr)),
        "min": float(np.min(arr)),
        "max": float(np.max(arr)),
        "p01": float(np.percentile(arr, 1)),
        "p05": float(np.percentile(arr, 5)),
        "p25": float(np.percentile(arr, 25)),
        "p50": float(np.percentile(arr, 50)),
        "p75": float(np.percentile(arr, 75)),
        "p95": float(np.percentile(arr, 95)),
        "p99": float(np.percentile(arr, 99)),
        "skewness": float(pd.Series(arr).skew())
    })

numeric_summary_df = pd.DataFrame(numeric_summary).sort_values("column")
display(numeric_summary_df.head(20))
numeric_summary_df.to_csv("../data/processed/ames_numeric_summary.csv", index=False)

In [ ]:
cat_summary = []

for col in cat_cols:
    mode_series = df[col].mode(dropna=True)
    cat_summary.append({
        "column": col,
        "n_unique": int(df[col].nunique(dropna=True)),
        "most_frequent": mode_series.iloc[0] if len(mode_series) > 0 else np.nan,
        "missing_count": int(df[col].isna().sum()),
        "missing_pct": round(df[col].isna().mean() * 100, 2)
    })

cat_summary_df = pd.DataFrame(cat_summary).sort_values("n_unique", ascending=False)
display(cat_summary_df.head(30))
cat_summary_df.to_csv("../data/processed/ames_categorical_summary.csv", index=False)

## Why categorical profiling?
Housing data lo categories chala important:
- neighborhood
- zoning
- house style
- quality columns
- garage finish

In [ ]:
selected_num_cols = [
    c for c in [
        "saleprice", "grlivarea", "lotarea", "totalbsmtsf",
        "1stflrsf", "2ndflrsf", "garagearea", "yearbuilt",
        "overallqual", "overallcond"
    ] if c in df.columns
]

df[selected_num_cols].hist(figsize=(16, 10), bins=30)
plt.suptitle("Selected Numerical Distributions", y=1.02, fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
for col in selected_num_cols:
    plt.figure(figsize=(10, 3.5))
    sns.boxplot(x=df[col])
    plt.title(f"Boxplot - {col}")
    plt.tight_layout()
    plt.show()

In [ ]:
selected_cat_cols = [
    c for c in [
        "neighborhood", "mszoning", "housestyle",
        "bldgtype", "exterqual", "kitchenqual", "garagefinish"
    ] if c in df.columns
]

for col in selected_cat_cols:
    print(f"\n===== {col} =====")
    display(df[col].value_counts(dropna=False).head(20))

In [ ]:
for col in selected_cat_cols:
    plt.figure(figsize=(11, 5))
    order = df[col].value_counts(dropna=False).index[:15]
    sns.countplot(data=df, x=col, order=order)
    plt.xticks(rotation=45, ha="right")
    plt.title(f"Countplot - {col}")
    plt.tight_layout()
    plt.show()

In [ ]:
corr_with_target = (
    df[num_cols]
    .corr(numeric_only=True)[target_col]
    .sort_values(ascending=False)
    .to_frame("corr_with_saleprice")
)

display(corr_with_target.head(20))
display(corr_with_target.tail(20))

In [ ]:
scatter_cols = [c for c in ["grlivarea", "totalbsmtsf", "garagearea", "yearbuilt", "overallqual"] if c in df.columns]

for col in scatter_cols:
    plt.figure(figsize=(7, 5))
    sns.scatterplot(data=df, x=col, y=target_col)
    plt.title(f"{col} vs {target_col}")
    plt.tight_layout()
    plt.show()

In [ ]:
corr_stats = []

for col in scatter_cols:
    temp = df[[col, target_col]].dropna()
    if len(temp) > 2:
        pr, pp = pearsonr(temp[col], temp[target_col])
        sr, sp = spearmanr(temp[col], temp[target_col])
        corr_stats.append({
            "column": col,
            "pearson_corr": pr,
            "pearson_pvalue": pp,
            "spearman_corr": sr,
            "spearman_pvalue": sp
        })

corr_stats_df = pd.DataFrame(corr_stats).sort_values("pearson_corr", ascending=False)
display(corr_stats_df)

## Why Pearson and Spearman?
- **Pearson** = linear relation
- **Spearman** = monotonic rank-based relation

In [ ]:
if "neighborhood" in df.columns:
    neighborhood_summary = (
        df.groupby("neighborhood", as_index=False)
          .agg(
              house_count=(target_col, "size"),
              avg_saleprice=(target_col, "mean"),
              median_saleprice=(target_col, "median")
          )
          .sort_values("avg_saleprice", ascending=False)
    )

    display(neighborhood_summary.head(20))
    neighborhood_summary.to_csv("../data/processed/neighborhood_price_summary.csv", index=False)

In [ ]:
if "overallqual" in df.columns:
    quality_summary = (
        df.groupby("overallqual", as_index=False)
          .agg(
              house_count=(target_col, "size"),
              avg_saleprice=(target_col, "mean"),
              median_saleprice=(target_col, "median")
          )
          .sort_values("overallqual")
    )

    display(quality_summary)
    quality_summary.to_csv("../data/processed/quality_price_summary.csv", index=False)

In [ ]:
if "neighborhood" in df.columns:
    top_neighborhoods = df["neighborhood"].value_counts().head(15).index
    plt.figure(figsize=(14, 6))
    sns.boxplot(data=df[df["neighborhood"].isin(top_neighborhoods)], x="neighborhood", y=target_col)
    plt.xticks(rotation=45, ha="right")
    plt.title("Neighborhood vs SalePrice")
    plt.tight_layout()
    plt.show()

if "overallqual" in df.columns:
    plt.figure(figsize=(10, 5))
    sns.boxplot(data=df, x="overallqual", y=target_col)
    plt.title("OverallQual vs SalePrice")
    plt.tight_layout()
    plt.show()

In [ ]:
top_corr_cols = corr_with_target["corr_with_saleprice"].abs().sort_values(ascending=False).head(12).index.tolist()

plt.figure(figsize=(12, 8))
sns.heatmap(df[top_corr_cols].corr(numeric_only=True), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Top Numerical Correlations")
plt.tight_layout()
plt.savefig("../reports/figures/correlation_heatmap.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
skew_report = (
    df[num_cols]
    .skew()
    .sort_values(ascending=False)
    .to_frame("skewness")
)

display(skew_report.head(30))
skew_report.to_csv("../data/processed/ames_skew_report.csv")

In [ ]:
outlier_rows = []

for col in num_cols:
    arr = df[col].dropna().to_numpy()
    if arr.size == 0:
        continue

    q1 = np.percentile(arr, 25)
    q3 = np.percentile(arr, 75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outlier_count = int(((arr < lower) | (arr > upper)).sum())
    outlier_pct = round((outlier_count / len(arr)) * 100, 2)

    outlier_rows.append({
        "column": col,
        "q1": q1,
        "q3": q3,
        "iqr": iqr,
        "lower_bound": lower,
        "upper_bound": upper,
        "outlier_count": outlier_count,
        "outlier_pct": outlier_pct
    })

outlier_report = pd.DataFrame(outlier_rows).sort_values("outlier_pct", ascending=False)
display(outlier_report.head(30))
outlier_report.to_csv("../data/processed/ames_outlier_report.csv", index=False)

## Why outlier analysis?
Regression lo outliers slope ni distort cheyyachu.  
Especially huge homes or unusual prices important.

In [ ]:
if {"grlivarea", "saleprice"}.issubset(df.columns):
    suspicious_large = df[(df["grlivarea"] > 4000)].sort_values("saleprice", ascending=False)
    display(suspicious_large[["id", "grlivarea", "saleprice"]].head(20) if "id" in suspicious_large.columns else suspicious_large[["grlivarea", "saleprice"]].head(20))

In [ ]:
if "grlivarea" in df.columns:
    temp = df[[target_col, "grlivarea"]].dropna().copy()
    temp["log_saleprice"] = np.log1p(temp[target_col])
    temp["log_grlivarea"] = np.log1p(temp["grlivarea"])

    plt.figure(figsize=(7, 5))
    sns.scatterplot(data=temp, x="log_grlivarea", y="log_saleprice")
    plt.title("log(GrLivArea) vs log(SalePrice)")
    plt.tight_layout()
    plt.show()

In [ ]:
high_missing_cols = missing_report[missing_report["missing_pct"] > 30].index.tolist()
moderate_missing_cols = missing_report[(missing_report["missing_pct"] > 0) & (missing_report["missing_pct"] <= 30)].index.tolist()

print("High missing columns (>30%):", high_missing_cols)
print("\nModerate missing columns (0-30%):", moderate_missing_cols[:25])

In [ ]:
df_clean = df.copy()
drop_cols = [col for col in high_missing_cols if col not in ["saleprice", "log_saleprice"]]
print("Columns to drop:", drop_cols)

df_clean = df_clean.drop(columns=drop_cols, errors="ignore")
print("Shape after dropping high-missing columns:", df_clean.shape)

In [ ]:
num_cols_clean = df_clean.select_dtypes(include=[np.number]).columns.tolist()
cat_cols_clean = df_clean.select_dtypes(include=["object"]).columns.tolist()

print("Numerical columns after drop:", len(num_cols_clean))
print("Categorical columns after drop:", len(cat_cols_clean))

In [ ]:
for col in num_cols_clean:
    if df_clean[col].isna().sum() > 0:
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

In [ ]:
for col in cat_cols_clean:
    if df_clean[col].isna().sum() > 0:
        df_clean[col] = df_clean[col].fillna("Unknown")

In [ ]:
for col in cat_cols_clean:
    df_clean[col] = df_clean[col].astype(str).str.strip()

In [ ]:
final_missing = df_clean.isna().sum().sort_values(ascending=False)
display(final_missing.head(20))
print("Total remaining missing values:", int(final_missing.sum()))

In [ ]:
df_clean.to_csv("../data/processed/ames_cleaned.csv", index=False)
print("Saved cleaned CSV: ../data/processed/ames_cleaned.csv")

## Why save cleaned CSV before modeling?
Exactly as discussed:
raw → preprocess → EDA → cleaned CSV → then modeling

In [ ]:
if {"grlivarea", "saleprice", "overallqual"}.issubset(df.columns):
    fig = px.scatter(
        df,
        x="grlivarea",
        y="saleprice",
        color="overallqual",
        title="GrLivArea vs SalePrice"
    )
    fig.write_html("../reports/figures/grlivarea_vs_saleprice.html")
    fig.show()

In [ ]:
if "neighborhood" in df.columns:
    fig = px.box(
        df,
        x="neighborhood",
        y="saleprice",
        title="Neighborhood vs SalePrice"
    )
    fig.update_layout(xaxis_tickangle=-45)
    fig.write_html("../reports/figures/neighborhood_vs_price.html")
    fig.show()

In [ ]:
if "overallqual" in df.columns:
    fig = px.box(
        df,
        x="overallqual",
        y="saleprice",
        title="OverallQual vs SalePrice"
    )
    fig.write_html("../reports/figures/quality_vs_price.html")
    fig.show()

In [ ]:
insights = '''
# Ames Housing Regression EDA Insights

1. The target variable `SalePrice` is right-skewed, and log transformation produces a more stable distribution.
2. Multiple columns contain missing values, and some high-missing columns were removed for a strong first-pass baseline dataset.
3. Numerical features such as `OverallQual`, `GrLivArea`, `GarageArea`, and `TotalBsmtSF` show strong relationships with `SalePrice`.
4. Some very large homes appear as potential outliers and should be examined carefully before model fitting.
5. Neighborhood-level pricing differences are substantial and likely useful for modeling.
6. Quality-related features appear strongly associated with price and should be retained for downstream models.
7. Median imputation for numerical columns and `Unknown` filling for categorical columns produced a clean modeling-ready dataset.
8. A cleaned CSV was saved for the next phase: regression model building.
'''

with open("../reports/insights.md", "w", encoding="utf-8") as f:
    f.write(insights)

print(insights)